# Setup

In [1]:
import io
import requests
import rasterio
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.wkt import loads

import sys
if sys.version_info < (3, 9):
    from importlib_resources import files
else:
    from importlib.resources import files

from beak.utilities.io import check_path
from beak.utilities.conversions import create_binary_raster


In [2]:
# Set base paths and files
BASE_PATH = files("beak.data")

EPSG_CODE = 102008
RESOLUTION = 500
BASE_SPATIAL = str(EPSG_CODE) + "_" + str(RESOLUTION)
BASE_EXTENT = "laculi_southwest"

BASE_RASTER = BASE_PATH / "BASE_RASTERS" / str("EPSG_" + str(EPSG_CODE) + "_RES_" + str(RESOLUTION) + "_" + BASE_EXTENT + ".tif")
base_raster = rasterio.open(BASE_RASTER)

# Points file and query to select relevant mineral occurences
PATH_LABELS = BASE_PATH / "RAW" / "mineral_deposits" / "Lacustrine_Lithium" / "TA2_Pre_Hack_12M" / "set_20240725"
check_path(PATH_LABELS)

# Set the output file
PATH_ROOT = BASE_PATH / "PROCESSED" / str("regional" + "_" + BASE_EXTENT + "_" + BASE_SPATIAL)
PATH_EXPORT = PATH_ROOT / "labels" / str("TA2_240725_FILTERED_RANDOM_PT" + ".tif")
OUT_FILE = PATH_EXPORT

print(f"Output raster: {OUT_FILE}")

Output raster: S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\PROCESSED\regional_laculi_southwest_102008_500\labels\TA2_240725_FILTERED_RANDOM_PT.tif


# Create raster

In [12]:
mineral_sites = gpd.read_file(PATH_LABELS / "mineral_sites_random_points.gpkg", layer="random_points_merge")


In [13]:
data = mineral_sites.copy()
data = data.explode(ignore_index=True)
data = data[~data.is_empty]
data = data.reset_index(drop=True)
data = data.to_crs(base_raster.crs)


In [16]:
labels_array = create_binary_raster(data, base_raster, all_touched=False, same_shape=True, fill_negatives=True, out_file=OUT_FILE)
print(f"Number of positive training labels: {np.sum(labels_array==1)}")

Number of positive training labels: 98
